# Data Creation

**One input panel in, one enriched panel out, covering all products.** The read, the month index, and the write are **global** steps (top and bottom). Each product section only *adds* columns to the shared dataframe via `src/rollups.py` — no functions in sections. The single output path is what every product config's `table` points to.

## Global setup — read the panel (single input)
One HDFS read; one row per CIF per month. `add_month_index` adds a contiguous integer month so the rollup windows count in whole months.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / 'configs' / '_schema.py').exists())
sys.path.insert(0, str(ROOT))
from pyspark.sql import SparkSession, Window, functions as F
spark = SparkSession.builder.getOrCreate()

from src.rollups import add_month_index, forward_rollup, backward_rollup, binary_target

INPUT_PATH  = 'hdfs:///retail/monthly_panel'                # single input (all products)
OUTPUT_PATH = 'hdfs:///retail/snapshots/modelling_panel'    # single output (all products)

df = spark.read.parquet(INPUT_PATH)
df = add_month_index(df, month_col='snapshot_month')
df.printSchema()

## 1. FX Activation
Adds trailing features, a currently-active flag, and the forward `fx_activation_target`.

### Backward rollups — trailing features + currently-active flag
Backward windows **end at and include the current month** (trailing N months). `is_fx_active` = any FX activity in the last 12 months — this is the column the modelling config's `eligibility_expr` filters on (`is_fx_active = 0`).

*Note:* because FX-activation eligibility keeps only FX-dormant clients, an FX-activity feature like `fx_txn_count_3m` is constant (0) for that population, so it is created here as a rollup example but not used in the FX activation config. It is useful for products (e.g. reactivation) whose eligible population can have prior FX activity.

In [ ]:
# trailing feature: FX transactions in the last 3 months (incl. current)
df = backward_rollup(df, id_col='cif', value_col='fx_txn', months=3,
                     out_col='fx_txn_count_3m', agg='sum')

# currently-active flag: any FX activity in the last 12 months (incl. current)
df = backward_rollup(df, id_col='cif', value_col='fx_txn', months=12,
                     out_col='fx_activity_12m', agg='sum')
df = binary_target(df, signal_col='fx_activity_12m', threshold=1,
                   out_col='is_fx_active')

### Forward rollup — the target
Aggregate the *next* 3 months and threshold. Activation = at least one qualifying FX transaction in (M, M+3]. The strictly-forward window is what separates label from features (which end at M).

In [ ]:
df = forward_rollup(df, id_col='cif', value_col='fx_txn', months=3,
                    out_col='fx_fwd_txn_3m', agg='sum')
df = binary_target(df, signal_col='fx_fwd_txn_3m', threshold=1,
                   out_col='fx_activation_target')

# panel-edge guard: null the target where the full forward window is missing
w = Window.partitionBy('cif').orderBy('month_idx').rangeBetween(1, 3)
df = df.withColumn('_n', F.count('month_idx').over(w))
df = df.withColumn('fx_activation_target',
                   F.when(F.col('_n') >= 3, F.col('fx_activation_target'))).drop('_n')

## 2. FX Reactivation *(next product — same pattern, new columns)*
Each further product repeats the pattern with its own columns/window/threshold. No new globals, no new functions — only column additions to the shared `df`.

In [ ]:
# df = backward_rollup(df, 'cif', 'fx_txn', 6, 'fx_txn_count_6m', agg='sum')
# df = forward_rollup(df, 'cif', 'fx_txn', 3, 'fx_react_fwd_3m', agg='sum')
# df = binary_target(df, 'fx_react_fwd_3m', 1, 'fx_reactivation_target')
pass

## Global finish — sanity check + write (single output)
Eyeball each product's target rate at the observation month, then write once.

In [ ]:
(df.filter(F.col('snapshot_month') == '2026-03-01')
   .agg(F.avg('fx_activation_target').alias('fx_base_rate'),
        F.avg('is_fx_active').alias('pct_active'),
        F.count('*').alias('rows')).show())

df.write.mode('overwrite').parquet(OUTPUT_PATH)